In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

/opt/conda/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
# caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
# culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")
golden_gate_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_transit_ridership.xlsx")
long_beach_transit_ridership=  pd.read_excel(f"{GCS_FILE_PATH}/long_beach_transit_ridership.xlsx")
# octa_ridership = pd.read_excel(f"{GCS_FILE_PATH}/octa_ridership.xlsx")
# omni_trans_ridership= pd.read_excel(f"{GCS_FILE_PATH}/omni_trans_ridership.xlsx")
riverside_transit_ridership= pd.read_excel(f"{GCS_FILE_PATH}/riverside_transit_ridership.xlsx")
# sacrt_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sacrt_bus_ridership.xlsx")
samtrans_ridership = pd.read_excel(f"{GCS_FILE_PATH}/samtrans_ridership.xlsx")
# santa_cruz_metro_ridership = pd.read_excel(f"{GCS_FILE_PATH}/santa_cruz_metro_ridership.xlsx")
# sbmtd_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sbmtd_ridership.xlsx")
sdmts_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sdmts_ridership.xlsx")
# sunline_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sunline_transit_ridership.xlsx")


In [5]:
def add_day_type(df, date_col, output_col='day_type'):
    """
    Classify each row as 'Weekday', 'Saturday', or 'Sunday' based on a specified date column.
    This is useful when daily datasets include labels such as 'Holiday' or when no day_type
    column exists. The function converts the date column, identifies the day of week, and
    standardizes the day-type classification accordingly. Either a start_date or end_date
    column can be used for datasets with daily reporting.
    """
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [40]:
def count_service_days(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum(d.weekday() < 5 for d in dates)
    elif row.day_type == "Saturday":
        return sum(d.weekday() == 5 for d in dates)
    elif row.day_type == "Sunday":
        return sum(d.weekday() == 6 for d in dates)

In [6]:
bart_ridership = add_day_type(bart_ridership, date_col='start_date')
bart_ridership.sample(5)

,record_id,dataset_id,organization_name,service_name,stop_id,stop_name,stop_lat,stop_lon,daily_alightings,daily_boardings,daily_total_ridership,start_date,end_date,daily_ridership_basis,day_type
17041,CC44C7507C2D5931,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,908309,Antioch,37.995373,-121.780346,1565,1508,NaN,2025-04-18,2025-04-18,reported_daily,Weekday
2897,57C750A1C1566B8C,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,901409,Civic Center / UN Plaza,37.779408,-122.413826,10304,11647,NaN,2025-08-28,2025-08-28,reported_daily,Weekday
4659,A75B0B6AB26CF42A,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,901909,Daly City,37.706259,-122.468908,4711,4593,NaN,2025-05-29,2025-05-29,reported_daily,Weekday
4800,C181BE06763E0D03,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,902109,Lake Merritt,37.797322,-122.265247,1647,1547,NaN,2025-07-19,2025-07-19,reported_daily,Saturday
12888,F64F49735752C690,011CF30F49575609,San Francisco Bay Area Rapid Transit District,Bay Area Rapid Transit,904609,Richmond,37.936758,-122.353047,1200,1194,NaN,2025-08-31,2025-08-31,reported_daily,Sunday


In [65]:
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")

In [66]:
big_blue_bus_ridership.columns = big_blue_bus_ridership.columns.str.lower()
big_blue_bus_ridership['service_day'] = big_blue_bus_ridership['service_day'].str.strip().str.capitalize()


big_blue_bus_ridership = big_blue_bus_ridership.rename(columns={
    'route_number': 'route_short_name',
    'service_day': 'day_type',
    'average_daily_boardings':'average_daily_boardings_period',
    'average_daily_alightings':'average_daily_alightings_period'    
})


# Sum boardings/alightings across directions within same stop & period
total_big_blue_bus_ridership = (
    big_blue_bus_ridership
    .groupby(['day_type','route_short_name','route_name',
              'stop_id','stop_name','stop_lat','stop_lon',
              'start_date','end_date'])
    .agg({
        'average_daily_boardings_period':'sum',
        'average_daily_alightings_period':'sum'
    })
    .reset_index()
)




In [68]:
big_blue_bus_ridership['start_date'] = pd.to_datetime(big_blue_bus_ridership['start_date'])
big_blue_bus_ridership['end_date'] = pd.to_datetime(big_blue_bus_ridership['end_date'])

total_big_blue_bus_ridership["n_days"] = total_big_blue_bus_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 4 different time periods
averaged_total_big_blue_bus_ridership = (
    total_big_blue_bus_ridership
    .groupby(['day_type','route_short_name','route_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'avg_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days']),
        'avg_daily_alightings': np.average(x['average_daily_alightings_period'], weights=x['n_days'])
    }))
    .reset_index()
)


# Set the ridership basis flag
averaged_total_big_blue_bus_ridership["daily_ridership_basis"] = "reported_avg_daily"


averaged_total_big_blue_bus_ridership.sample(5)

/tmp/ipykernel_3463/4108272206.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,day_type,route_short_name,route_name,stop_id,stop_lat,stop_lon,avg_daily_boardings,avg_daily_alightings
2996,Weekday,44,SMC Campus Connector,2954,34.018194,-118.478955,2.906641,0.626502
449,Saturday,9,Pacific Palisades,1059,34.026029,-118.506576,1.659510,0.068578
1565,Sunday,18,UCLA - Abbot Kinney,2216,34.052506,-118.470015,5.608202,7.724668
1893,Weekday,3,Lincoln Blvd/LAX,2692,33.993979,-118.452311,88.270620,135.292853
2797,Weekday,23,Lincoln Blvd/LAX Rapid,2238,33.945343,-118.379127,0.521533,14.971439


In [7]:
# Add day_type from the date column
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col="date")

# Grouping columns
group_cols = [
    "date", "route_short_name", "stop_code", "stop_lat", "stop_lon",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_ridership_foothill = (
    foothill_transit_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("boardings", "sum"),
        daily_alightings=("alightings", "sum")
    )
)

# Set the ridership basis flag
total_ridership_foothill["daily_ridership_basis"] = "reported_daily"

total_ridership_foothill.head(5)


,date,route_short_name,stop_code,stop_lat,stop_lon,start_date,end_date,day_type,daily_boardings,daily_alightings,daily_ridership_basis
0,2024-07-01,178,23,34.034964,-117.919263,2024-07-01,2024-07-01,Weekday,1,0,reported_daily
1,2024-07-01,178,129,33.990165,-117.888915,2024-07-01,2024-07-01,Weekday,13,3,reported_daily
2,2024-07-01,178,555,34.030813,-117.914021,2024-07-01,2024-07-01,Weekday,56,40,reported_daily
3,2024-07-01,178,556,34.031712,-117.915058,2024-07-01,2024-07-01,Weekday,47,44,reported_daily
4,2024-07-01,178,599,34.026486,-117.903267,2024-07-01,2024-07-01,Weekday,1,0,reported_daily


In [8]:
# Add day_type from the date column
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col="Date")

fresno_area_express_ridership["daily_ridership_basis"] = "reported_daily"


fresno_area_express_ridership = fresno_area_express_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopLabel': 'stop_name',
    'ProjectedBoarding': 'daily_boardings',
    'ProjectedAlighting': 'daily_alightings'
})

fresno_area_express_ridership.head(5)

,Date,stop_id,stop_name,daily_boardings,daily_alightings,start_date,end_date,day_type,daily_ridership_basis
0,2024-09-01,5,NE BRAWLEY - SHIELDS,44.691729,29.748092,2024-09-01,2024-09-01,Sunday,reported_daily
1,2024-09-01,6,SE SHAW - BRAWLEY,7.000000,0.000000,2024-09-01,2024-09-01,Sunday,reported_daily
2,2024-09-01,7,SW SHAW - WEST,20.000000,20.000000,2024-09-01,2024-09-01,Sunday,reported_daily
3,2024-09-01,8,SE SHAW - BLACKSTONE,79.000000,17.000000,2024-09-01,2024-09-01,Sunday,reported_daily
4,2024-09-01,9,SE SHAW - FIRST,63.000000,29.000000,2024-09-01,2024-09-01,Sunday,reported_daily


In [14]:
# Add day_type from the date column
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col="Date")

group_cols = [
    "Date", "day_type", "Stop", "start_date", "end_date"]

# Sum AVG On 
total_golden_gate_park_shuttle_ridership = (golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Ridership", "sum"),
    )
)

total_golden_gate_park_shuttle_ridership["daily_ridership_basis"] = "reported_daily"

total_golden_gate_park_shuttle_ridership = total_golden_gate_park_shuttle_ridership.rename(columns={
    'Stop': 'stop_name',
    'Date': 'date'
})

total_golden_gate_park_shuttle_ridership.sample(5)

,Date,day_type,stop_name,start_date,end_date,daily_boardings,daily_ridership_basis
2908,2024-12-09,Weekday,JFK Gateway EB,2024-12-09,2024-12-09,7,reported_daily
1686,2024-10-02,Weekday,Music Concourse,2024-10-02,2024-10-02,0,reported_daily
5869,2025-05-23,Weekday,10th Ave/ De Young WB,2025-05-23,2025-05-23,0,reported_daily
6104,2025-06-05,Weekday,8th Ave EB,2025-06-05,2025-06-05,28,reported_daily
4141,2025-02-16,Sunday,10th Ave/ De Young WB,2025-02-16,2025-02-16,0,reported_daily


In [75]:
golden_gate_transit_ridership.sample(5)

,OPERATION_DATE,ROUTE,DIRECTION,STOP_NUMBER,STOP_NAME,BOARDINGS,ALIGHTINGS,date,start_date,end_date,day_type
694,02-SEP-25,132,South,80035,VTP 101 SB @ Sir Francis Drake (80035),0,0,2025-09-02,2025-09-02,2025-09-02,Weekday
4740,08-SEP-25,132,South,40055,The Embarcadero & Bay St (40055),0,4,2025-09-08,2025-09-08,2025-09-08,Weekday
9298,14-SEP-25,580,West,41092,Francisco Blvd E & Morphew St (41092),1,0,2025-09-14,2025-09-14,2025-09-14,Sunday
3030,05-SEP-25,114,North,40148,Miller Ave & Reed St (40148),0,6,2025-09-05,2025-09-05,2025-09-05,Weekday
5793,09-SEP-25,164,North,40838,Petaluma Blvd S & G St (40838),0,4,2025-09-09,2025-09-09,2025-09-09,Weekday


In [76]:
# Add day_type from the date column
golden_gate_transit_ridership = add_day_type(golden_gate_transit_ridership, date_col="date")


golden_gate_transit_ridership = golden_gate_transit_ridership.rename(columns={
    'STOP_NUMBER': 'stop_id',
    'STOP_NAME': 'stop_name',
    'ROUTE': 'route_short_name',
})

# Remove everything in parentheses (including the parentheses) at the end of the string
golden_gate_transit_ridership['stop_name'] = golden_gate_transit_ridership['stop_name'].str.replace(r'\s*\(.*\)$', '', regex=True)

group_cols = [
    "day_type", "route_short_name", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_golden_gate_transit_ridership = (golden_gate_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("BOARDINGS", "sum"),
        daily_alightings=("ALIGHTINGS", "sum")
    )
)

total_golden_gate_transit_ridership["daily_ridership_basis"] = "reported_daily"

total_golden_gate_transit_ridership.sample(5)

,day_type,route_short_name,stop_id,stop_name,end_date,start_date,daily_boardings,daily_alightings,daily_ridership_basis
7409,Weekday,130,40107,Alexander Ave & East Rd (40107),2025-09-30,2025-09-30,0,1,reported_daily
17032,Weekday,172,80003,VTP 101 NB @ North End of Bridge (80003),2025-09-02,2025-09-02,0,0,reported_daily
10857,Weekday,150,40078,Van Ness Ave & Vallejo St (40078),2025-09-03,2025-09-03,2,9,reported_daily
7560,Weekday,130,40267,Lucky Dr Bus Pad (40267),2025-09-25,2025-09-25,9,5,reported_daily
16113,Weekday,172,40051,Battery St & Sacramento St (40051),2025-09-09,2025-09-09,0,27,reported_daily


In [10]:
long_beach_transit_ridership["daily_ridership_basis"] = "reported_avg_daily"

group_cols = [
    "DayType", "Route", "StopID", "StopName", "end_date", "start_date", "daily_ridership_basis"]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Boardings", "sum"),
        daily_alightings=("Alightings", "sum")
    )
)

total_long_beach_transit_ridership = total_long_beach_transit_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopName': 'stop_name',
    'DayType': 'day_type',
    'Route': 'route_short_name',
})


total_long_beach_transit_ridership.head(5)

,day_type,route_short_name,stop_id,stop_name,end_date,start_date,daily_ridership_basis,daily_boardings,daily_alightings
0,Saturday,1,2001,2727 Del Amo Blvd N,2025-06-30,2024-07-01,reported_avg_daily,2.582828,0.000000
1,Saturday,1,2002,2660 Del Amo Blvd S,2025-06-30,2024-07-01,reported_avg_daily,0.000000,0.000000
2,Saturday,1,2003,Del Amo & Rancho Way NW,2025-06-30,2024-07-01,reported_avg_daily,2.282203,4.831939
3,Saturday,1,2004,Del Amo & Fordyce SW,2025-06-30,2024-07-01,reported_avg_daily,5.977199,2.219394
4,Saturday,1,2005,Del Amo & Wilmington NW,2025-06-30,2024-07-01,reported_avg_daily,3.527042,5.859201


In [11]:
riverside_transit_ridership = add_day_type(riverside_transit_ridership, date_col='date')

group_cols = [
    "date", "Stop ID", "Route", "Longitude", "Latitude", "start_date", 
    "end_date", "day_type"]

# Sum AVG On and AVG Off
total_riverside_transit_ridership = (riverside_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    daily_boardings = ('ridership', 'sum'))
)

total_riverside_transit_ridership = total_riverside_transit_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_short_name',
    'Latitude': 'stop_lat',
    'Longitude': 'stop_lon'

})

total_riverside_transit_ridership["daily_ridership_basis"] = "reported_daily"
total_riverside_transit_ridership.sample(5)


,date,stop_id,route_short_name,stop_lon,stop_lat,start_date,end_date,day_type,daily_boardings,daily_ridership_basis
49862,2025-01-30,3406,79,-117.139912,33.552444,2025-01-30,2025-01-30,Weekday,1,reported_daily
177645,2025-04-13,1029,1,-117.438848,33.927260,2025-04-13,2025-04-13,Sunday,10,reported_daily
394967,2025-09-24,1996,11,-117.243712,33.931872,2025-09-24,2025-09-24,Weekday,1,reported_daily
268031,2025-07-12,1794,15,-117.460664,33.902808,2025-07-12,2025-07-12,Saturday,1,reported_daily
269707,2025-07-13,1932,19,-117.226456,33.935324,2025-07-13,2025-07-13,Sunday,10,reported_daily


In [12]:
samtrans_ridership = add_day_type(samtrans_ridership, date_col='APC Date')

samtrans_ridership = samtrans_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_short_name',
    'Lat': 'stop_lat',
    'Lon': 'stop_lon',
    'APC Date': 'date',
    'Ons': 'daily_boardings',
    'Offs': 'daily_alightings',
})

samtrans_ridership["daily_ridership_basis"] = "reported_daily"
samtrans_ridership.sample(5)


,route_short_name,date,stop_id,Stop Name,daily_boardings,daily_alightings,stop_lat,stop_lon,day_type,start_date,end_date,daily_ridership_basis
55945,ECR,2025-08-22,341158,El Camino Real & State St,18,35,37.574062,-122.340286,Weekday,2025-08-22,2025-08-22,reported_daily
8296,120,2025-08-29,332314,Southgate Ave & Lincoln Ave,80,41,37.679637,-122.486793,Weekday,2025-08-29,2025-08-29,reported_daily
35083,295,2025-08-08,341620,256 37th Ave-San Mateo Medical Cent,1,0,37.531851,-122.300792,Weekday,2025-08-08,2025-08-08,reported_daily
19448,142,2025-08-25,335630,San Bruno BART-Bay 7 Outer Busway,41,40,37.641709,-122.198972,Weekday,2025-08-25,2025-08-25,reported_daily
33687,292,2025-08-30,331605,Bayshore Blvd & Fitzgerald Ave,2,31,37.727436,-122.401626,Saturday,2025-08-30,2025-08-30,reported_daily


In [13]:
group_cols = [
    "Route", "Route Name", "Stop ID",  "Stop Name", "day_type", "start_date", 
    "end_date"]

# Sum AVG On and AVG Off
total_sdmts_ridership = (sdmts_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Average On", "sum"),
        daily_alightings=("Average Off", "sum")
    )
)

total_sdmts_ridership["Route Name"] = total_sdmts_ridership["Route Name"].str.split(":", n=1).str[1]

total_sdmts_ridership = total_sdmts_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Route': 'route_short_name',
    'Route Name': 'route_name',
    'Stop Name': 'stop_name',
})

total_sdmts_ridership["daily_ridership_basis"] = "reported_avg_daily"

total_sdmts_ridership.head(10)

,route_short_name,route_name,stop_id,stop_name,day_type,start_date,end_date,daily_boardings,daily_alightings,daily_ridership_basis
0,1,Fashion Valley-La Mesa,10106,University Av & 10th Av,Saturday,2024-09-01,2025-01-25,16.513085,8.709722,reported_avg_daily
1,1,Fashion Valley-La Mesa,10106,University Av & 10th Av,Sunday,2024-09-01,2025-01-25,14.264056,8.324885,reported_avg_daily
2,1,Fashion Valley-La Mesa,10106,University Av & 10th Av,Weekday,2024-09-01,2025-01-25,26.228026,15.629832,reported_avg_daily
3,1,Fashion Valley-La Mesa,10111,University Av & Vermont St,Saturday,2024-09-01,2025-01-25,42.220322,14.795541,reported_avg_daily
4,1,Fashion Valley-La Mesa,10111,University Av & Vermont St,Sunday,2024-09-01,2025-01-25,29.279929,10.192920,reported_avg_daily
5,1,Fashion Valley-La Mesa,10111,University Av & Vermont St,Weekday,2024-09-01,2025-01-25,59.583915,17.940893,reported_avg_daily
6,1,Fashion Valley-La Mesa,10114,University Av & Richmond St,Saturday,2024-09-01,2025-01-25,9.976243,6.169591,reported_avg_daily
7,1,Fashion Valley-La Mesa,10114,University Av & Richmond St,Sunday,2024-09-01,2025-01-25,7.180691,8.281819,reported_avg_daily
8,1,Fashion Valley-La Mesa,10114,University Av & Richmond St,Weekday,2024-09-01,2025-01-25,14.963778,14.173884,reported_avg_daily
9,1,Fashion Valley-La Mesa,10129,El Cajon Bl & Alabama St,Saturday,2024-09-01,2025-01-25,7.576505,5.638085,reported_avg_daily
